In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS coreview")
spark.sql("CREATE SCHEMA IF NOT EXISTS coreview.bronze")

as we want to capture all the changes for every account we wont be checking for duplicates for this dimension

In [0]:
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
accounts_path = "/Volumes/coreview/landing/organizations_data/data/accounts"

In [0]:
accounts_df = spark.read.format('csv').option('header', 'true').load(accounts_path)

Adding timestamp columns

In [0]:
accounts_df = (accounts_df.withColumn("Ingestion_timestamp", F.current_timestamp())
                        .withColumn("Ingestion_date", F.to_date("Ingestion_timestamp"))
)

In [0]:
# display(accounts_df)

In [0]:
acc_table_name = "coreview.bronze.crm_accounts"
accounts_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").partitionBy('Ingestion_date').saveAsTable(acc_table_name)

In [0]:
accounts_df.count()

180

if we are doing incremental ingestion then we overwrite for initial run and then keep appending

In [0]:
# acc_table_name = "coreview.bronze.crm_accounts"

# if spark.catalog.tableExists(acc_table_name):
#     accounts_df.write.format("delta").mode("append").option("mergeSchema", "true").partitionBy('Ingestion_date').saveAsTable(acc_table_name)
# else:
#     accounts_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").partitionBy('Ingestion_date').saveAsTable(acc_table_name)


In [0]:
# %%sql
# select * from coreview.bronze.crm_accounts

as we want to capture all the changes for every organisation we wont be checking for duplicates for this dimension

In [0]:
organisations_path = '/Volumes/coreview/landing/organizations_data/data/organizations/'

the file name of the json has a date in it which is teh date showing the state of the organisation so capture it

In [0]:
org_df = (
    spark.read
    .option("multiLine", "false")
    .json(organisations_path + "*.jsonl")
    .withColumn("source_file_name", F.col("_metadata.file_name"))
)

In [0]:
display(org_df)

created_at,crm_ref,license,org_id,region,status,tenant_name,source_file_name
2025-03-25T17:34:00Z,001FXqc67PSEoGvPIC,"List(List(audit), starter, 1000)",0070366f29d4baa30aecd4bf,US,active,Harbor Insurance AU,2026-02-24.jsonl
2025-03-17T00:31:00Z,001jUw1SWAw7Uxz,"List(List(backup), standard, 100)",0367b50f9264c4019b39fc28,CA,active,Highland Legal,2026-02-24.jsonl
2025-10-11T16:53:00Z,001uyFt0STZozPa,"List(List(audit, reports), starter, 10)",040f2aec3fcc9a8ea7d33a51,AU,active,Westgate Retail,2026-02-24.jsonl
2024-10-01T01:23:00Z,001miHb18TUzEoK,"List(List(audit, backup), enterprise, 500)",04a836d16bf160863057a5f6,AU,active,Starling Energy,2026-02-24.jsonl
2024-08-02T17:49:00Z,0013fRR0TINRKOB,"List(List(), starter, 500)",07455d260415f3ed1a9f7130,AU,active,Juniper Logistics,2026-02-24.jsonl
2026-01-27T00:57:00Z,0019rawhwiRcUA0,"List(List(backup, reports), starter, 2000)",075e8f602fb1775b362e67fb,US,active,Summit Health,2026-02-24.jsonl
2024-08-17T13:12:00Z,001rz3hh3w7i8y1,"List(List(audit, reports), standard, 50)",091c578e9022ba711b0b9f25,CA,active,Sterling Foods,2026-02-24.jsonl
2025-03-02T12:13:00Z,0011zZhPeWGPAkh,"List(List(audit, reports), enterprise, 50)",094cc970cd7cf30aa30ff992,CA,active,Cedar Foods,2026-02-24.jsonl
2024-02-02T02:16:00Z,001g3rn2nFZ5b4uBSA,"List(List(audit, backup), standard, 100)",0a495cb327b4d82cf8e9ebc0,EU,active,Cedar Media,2026-02-24.jsonl
2026-02-08T09:49:00Z,0015KugxFSxBaRFLDJ,"List(List(audit), starter, 500)",0c0faa81bc5d2e473d8981a7,AU,active,Westgate Finance,2026-02-24.jsonl


In [0]:
org_df.printSchema()

root
 |-- created_at: string (nullable = true)
 |-- crm_ref: string (nullable = true)
 |-- license: struct (nullable = true)
 |    |-- addons: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- plan: string (nullable = true)
 |    |-- seats: string (nullable = true)
 |-- org_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- status: string (nullable = true)
 |-- tenant_name: string (nullable = true)
 |-- source_file_name: string (nullable = false)



used Genie to build the below method

In [0]:
from pyspark.sql.types import StructType

def flatten_df(df):
    """Flatten nested structs into parent_child columns (arrays are left intact)."""
    while any(isinstance(df.schema[c].dataType, StructType) for c in df.columns):
        cols = []
        for c in df.columns:
            if isinstance(df.schema[c].dataType, StructType):
                for f in df.schema[c].dataType.fields:
                    cols.append(F.col(f"{c}.{f.name}").alias(f"{c}_{f.name}"))
            else:
                cols.append(F.col(c))
        df = df.select(*cols)
    return df

org_flat = flatten_df(org_df)
org_flat.printSchema()

root
 |-- created_at: string (nullable = true)
 |-- crm_ref: string (nullable = true)
 |-- license_addons: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- license_plan: string (nullable = true)
 |-- license_seats: string (nullable = true)
 |-- org_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- status: string (nullable = true)
 |-- tenant_name: string (nullable = true)
 |-- source_file_name: string (nullable = false)



In [0]:
display(org_flat)

created_at,crm_ref,license_addons,license_plan,license_seats,org_id,region,status,tenant_name,source_file_name
2025-03-25T17:34:00Z,001FXqc67PSEoGvPIC,List(audit),starter,1000,0070366f29d4baa30aecd4bf,US,active,Harbor Insurance AU,2026-02-24.jsonl
2025-03-17T00:31:00Z,001jUw1SWAw7Uxz,List(backup),standard,100,0367b50f9264c4019b39fc28,CA,active,Highland Legal,2026-02-24.jsonl
2025-10-11T16:53:00Z,001uyFt0STZozPa,"List(audit, reports)",starter,10,040f2aec3fcc9a8ea7d33a51,AU,active,Westgate Retail,2026-02-24.jsonl
2024-10-01T01:23:00Z,001miHb18TUzEoK,"List(audit, backup)",enterprise,500,04a836d16bf160863057a5f6,AU,active,Starling Energy,2026-02-24.jsonl
2024-08-02T17:49:00Z,0013fRR0TINRKOB,List(),starter,500,07455d260415f3ed1a9f7130,AU,active,Juniper Logistics,2026-02-24.jsonl
2026-01-27T00:57:00Z,0019rawhwiRcUA0,"List(backup, reports)",starter,2000,075e8f602fb1775b362e67fb,US,active,Summit Health,2026-02-24.jsonl
2024-08-17T13:12:00Z,001rz3hh3w7i8y1,"List(audit, reports)",standard,50,091c578e9022ba711b0b9f25,CA,active,Sterling Foods,2026-02-24.jsonl
2025-03-02T12:13:00Z,0011zZhPeWGPAkh,"List(audit, reports)",enterprise,50,094cc970cd7cf30aa30ff992,CA,active,Cedar Foods,2026-02-24.jsonl
2024-02-02T02:16:00Z,001g3rn2nFZ5b4uBSA,"List(audit, backup)",standard,100,0a495cb327b4d82cf8e9ebc0,EU,active,Cedar Media,2026-02-24.jsonl
2026-02-08T09:49:00Z,0015KugxFSxBaRFLDJ,List(audit),starter,500,0c0faa81bc5d2e473d8981a7,AU,active,Westgate Finance,2026-02-24.jsonl


In [0]:
org_flat = (org_flat.withColumn("Ingestion_timestamp", F.current_timestamp())
                        .withColumn("Ingestion_date", F.to_date("Ingestion_timestamp"))
)

if we are doing incremental ingestion then we overwrite for initial run and then keep appending

In [0]:
org_flat.printSchema()

root
 |-- created_at: string (nullable = true)
 |-- crm_ref: string (nullable = true)
 |-- license_addons: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- license_plan: string (nullable = true)
 |-- license_seats: string (nullable = true)
 |-- org_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- status: string (nullable = true)
 |-- tenant_name: string (nullable = true)
 |-- source_file_name: string (nullable = false)
 |-- Ingestion_timestamp: timestamp (nullable = false)
 |-- Ingestion_date: date (nullable = false)



In [0]:
# org_table_name = "coreview.bronze.organizations"

# if spark.catalog.tableExists(org_table_name):
#     org_flat.write.format("delta").mode("append").option("mergeSchema", "true").partitionBy('Ingestion_date').saveAsTable(org_table_name)
# else:
#     org_flat.write.format("delta").mode("overwrite").option("mergeSchema", "true").partitionBy('Ingestion_date').saveAsTable(org_table_name)


In [0]:
org_table_name = "coreview.bronze.organizations"
org_flat.write.format("delta").mode("overwrite").option("mergeSchema", "true").partitionBy('Ingestion_date').saveAsTable(org_table_name)

In [0]:
org_flat.count()

12125

In [0]:
# %%sql
# select * from coreview.bronze.organizations

In [0]:
signins_path = "/Volumes/coreview/landing/organizations_data/data/signins"

In [0]:
signins_df = spark.read.format('csv').option('header', 'true').load(signins_path)

In [0]:
signins_df.count()

302593

In [0]:
signins_df = (signins_df.withColumn("Ingestion_timestamp", F.current_timestamp())
                        .withColumn("Ingestion_date", F.to_date("Ingestion_timestamp"))
)

In [0]:
# signins_table_name = "coreview.bronze.signins"

# if spark.catalog.tableExists(signins_table_name):
#     signins_df.write.format("delta").mode("append").option("mergeSchema", "true").partitionBy('Ingestion_date').saveAsTable(signins_table_name)
# else:
#     signins_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").partitionBy('Ingestion_date').saveAsTable(signins_table_name)


In [0]:
signins_table_name = "coreview.bronze.signins"
signins_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").partitionBy('Ingestion_date').saveAsTable(signins_table_name)

In [0]:
# %%sql
# select * from coreview.bronze.signins